# Swarm Clock, Part 2c: confound controls and corrected calibration
Run in the SAME runtime right after Part 2b (uses R, D_primary, elig, assign_families, family_table, shingle_set, minhash, plant, perm_test, stat).
Pre-declared reading: if the real signal disappears under family x day strata, it is explained by between-day drift (task schedule), i.e. no evidence of a copying clock.

In [ ]:
def reps_with_strata(D, fam_ids, mode):
    X = D[D.family.isin(fam_ids)].copy()
    X["len"] = X.m.str.len()
    X = X.sort_values(["family","page_id","len","rev_id"], ascending=[True,True,False,True]).drop_duplicates(["family","page_id"])
    lt = X.groupby("family")["len"].transform(lambda s: pd.qcut(s.rank(method="first"), 3, labels=False) if len(s) >= 6 else 0).astype(int).astype(str)
    day = X.t.dt.strftime("%Y-%m-%d")
    X["stratum"] = {"length": lt, "day": day, "day_length": day + "_" + lt}[mode].values
    return X
def arrays_with_ids(X, k=3):
    arrs = family_arrays(X, k)
    return arrs
def per_family_table(X, arrs):
    rows = []
    for a in arrs:
        sub = X[X.family == a["f"]]
        dt_ = np.abs(a["t"][a["iu"][0]] - a["t"][a["iu"][1]])
        rho = np.corrcoef(a["rd"], rankdata(dt_))[0,1] if np.ptp(dt_) > 0 and np.ptp(a["rd"]) > 0 else np.nan
        rows.append({"family": a["f"], "reps": len(sub), "days": sub.t.dt.date.nunique(),
                     "strata": len(np.unique(a["strata"])), "rho": round(rho, 3),
                     "example": sub.sort_values("t").m.iloc[0][:60]})
    return pd.DataFrame(rows)

## 1. Real data: per-family rho and three stratifications

In [ ]:
RESULTS = []
for mode in ["length", "day", "day_length"]:
    X = reps_with_strata(D_primary, elig.index, mode)
    arrs = family_arrays(X)
    T, p, _ = perm_test(arrs, B=999)
    RESULTS.append({"data": "real", "strata": mode, "families": len(arrs), "T": round(T,3), "p": round(p,4)})
    if mode == "length": print(per_family_table(X, arrs).to_string(index=False))
print(pd.DataFrame(RESULTS).to_string(index=False))

## 2. Template ablation: remove the wiki's default new-page text, rebuild families, retest

In [ ]:
TEMPLATE = re.compile(r"Beschreibe hier die neue Seite\.?", re.I)
R_ab = R.assign(m=R.m.str.replace(TEMPLATE, " ", regex=True).str.strip())
R_ab = R_ab[R_ab.m.str.split().str.len() >= 8].reset_index(drop=True)
fam_ab = assign_families([minhash(shingle_set(t, 3)) for t in R_ab.m], 0.3)
D_ab, G_ab = family_table(R_ab, fam_ab)
elig_ab = G_ab[G_ab.blocks >= 15].sort_values("blocks", ascending=False)
print(elig_ab[["blocks","labels","span_h","days"]].to_string())
for mode in ["length", "day"]:
    X = reps_with_strata(D_ab, elig_ab.index, mode); arrs = family_arrays(X)
    T, p, _ = perm_test(arrs, B=999)
    RESULTS.append({"data": "real, template removed", "strata": mode, "families": len(arrs), "T": round(T,3), "p": round(p,4)})
    if mode == "length": print(per_family_table(X, arrs).to_string(index=False))
print(pd.DataFrame(RESULTS).to_string(index=False))

## 3. Corrected calibration: test ONLY the planted members (no contamination by real members)
Also reports the contaminated version (whole recovered family), which reproduces the anti-conservative behavior seen in Part 2b.

In [ ]:
def run_condition2(kind, p, B=199):
    P = plant(kind, p)
    A = pd.concat([R.assign(lin=-1), P], ignore_index=True)
    mhs = real_minhashes(3) + [minhash(shingle_set(t, 3)) for t in P.m]
    fam = assign_families(mhs, 0.3)
    D, g = family_table(A[["rev_id","page_id","label","t","m"]], fam)
    D["lin"] = A.lin.values
    res = []
    for li in range(P.lin.max()+1):
        mem = D[D.lin == li]; f = mem.family.mode().iloc[0]
        complete = (mem.family == f).mean()
        recovered = complete >= .8 and g.loc[f,"blocks"] >= 15 and g.loc[f,"days"] >= 3
        out = {"kind": kind, "p": p, "lin": li, "completeness": complete, "recovered": recovered,
               "contaminated_real_members": int(((D.family == f) & (D.lin == -1)).sum())}
        if recovered:
            only = D[(D.lin == li) & (D.family == f)]
            for mode in ["length", "day"]:
                out[f"p_{mode}"] = perm_test(family_arrays(reps_with_strata(only, [f], mode)), B=B)[1]
            out["p_contaminated"] = perm_test(family_arrays(reps_with_strata(D[D.family == f], [f], "length")), B=B)[1]
        res.append(out)
    return pd.DataFrame(res)
CAL2 = pd.concat([run_condition2(kind, p) for kind in ["clock", "star"] for p in [0.02, 0.05, 0.10]], ignore_index=True)
rate = lambda s: round(np.mean(s.dropna() < .05), 2) if s.notna().any() else np.nan
summary2 = CAL2.groupby(["kind","p"]).agg(recovery=("recovered","mean"), contaminated=("contaminated_real_members","median"),
    sig_length=("p_length", rate), sig_day=("p_day", rate), sig_contaminated=("p_contaminated", rate))
print(summary2.to_string()); CAL2.to_csv("data/calibration_corrected.csv", index=False)
pd.DataFrame(RESULTS).to_csv("data/temporal_tests_real.csv", index=False)

## Reading
- Valid test: `star` sig_length and sig_day near 0.05; `clock` high.
- Real signal under `length` but not under `day` means it is explained by between-day drift (task schedule or templates), not by an accumulating copying clock.
- Note: in `day` strata the clock lineages lose between-day contrast, so power drops; compare real vs clock power at the same strata.